# Data Explorer
Inspect the downloaded price data as an interactive table.
Use the **Open in Data Wrangler** button (appears above any DataFrame output) for a full spreadsheet view.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.float_format", "{:.2f}".format)

DATA_DIR = Path("../data/raw")

## Load Data

In [ ]:
# List available files
available = list(DATA_DIR.glob("*.parquet")) + list(DATA_DIR.glob("*.csv"))
print("Files in data/raw:")
for f in available:
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# ── Change "international" to "sp500" once you download that file ──
FILE = "international"

path = DATA_DIR / f"{FILE}.parquet"
df_raw = pd.read_parquet(path)

print(f"Shape : {df_raw.shape}")
print(f"Date range: {df_raw.index.min().date()} → {df_raw.index.max().date()}")
print(f"\nColumn levels:\n{df_raw.columns.tolist()[:10]} …")

In [ ]:
# Extract the Close price table — one column per ticker
close = df_raw["Close"]
print("Close price table  (rows=dates, cols=tickers):")
display(close.tail(10))

## Filter and Sort

In [ ]:
# Summary stats per ticker — useful to spot gaps (NaN) and data quality issues
summary = close.agg(["count", "min", "max", "mean", "std"]).T
summary.columns = ["trading_days", "min_price", "max_price", "mean_price", "std"]
summary = summary.sort_values("mean_price", ascending=False)
display(summary)

In [ ]:
# Filter by date range and pick specific tickers
START = "2022-01-01"
END   = "2024-12-31"
TICKERS = close.columns.tolist()  # or e.g. ["ASML.AS", "SAP.DE"]

subset = close.loc[START:END, TICKERS]
print(f"Filtered: {len(subset)} rows × {len(subset.columns)} tickers")
display(subset.head(20))

## Styled Table
Color gradient highlights high (green) and low (red) closing prices per column.

In [ ]:
# Show last 10 rows with a red→green color gradient per column
# (shows how each ticker's latest price compares to its own recent range)
sample = subset.tail(10)

styled = (
    sample.style
    .background_gradient(cmap="RdYlGn", axis=0)   # per column
    .format("{:.2f}")
    .set_caption(f"Adjusted Close Prices — {START} to {END}")
)
display(styled)

## Open in Data Wrangler (full spreadsheet view)
After running any cell that outputs a DataFrame, a **"Open in Data Wrangler"** button appears in the output toolbar.  
Click it for a resizable, sortable, filterable spreadsheet — no extra code needed.

You can also right-click any `.parquet` file in the Explorer panel → **Open in Data Wrangler** to view it directly.